In [ ]:
!pip install ollama
!pip install fitz
!pip install frontend
!pip install PyMuPDF
!pip install pyreadr
!pip install openpyxl
!pip install qwen_vl_utils
!pip install pillow
!pip install torch
!pip install torchvision
#!pip install tabula-py
#!pip install jpype1
!pip install pdfplumber
#!pip install camelot-py

In [ ]:
!pip install pdf2image

In [3]:
%cd ../../daan/Chloe-project/

/daan/Chloe-project


In [4]:
import sys
import base64
import os
import requests
import ollama
import fitz  # PyMuPDF
import io
import re
import pandas as pd 
import numpy as np 
import json
import time
import pyreadr
from openpyxl import Workbook, load_workbook
from pathlib import Path
import importlib
import functions
importlib.reload(functions)


SEED = 1

# Show all columns without truncation
pd.set_option("display.max_colwidth", None)  

# If you also want to see all rows/columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [5]:
import pdfplumber

study_id = "1835" #"2678"

pdf_path = os.path.join(f"extraction_papers/{study_id}.pdf")

extracted_text = ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:  # avoid None for empty pages
            extracted_text += text + "\n\n"  # add spacing between pages

In [ ]:
print(extracted_text)

In [ ]:
with open(f"training_set/markdown/{study_id}.md", 'r') as f: 
    markdown_text = f.read()

print(markdown_text)

In [434]:
# Example usage of pdf_to_images(): 
image_paths = pdf_to_images(study_id)
print(f"Saved {len(image_paths)} images:", image_paths)

Saved 12 images: ['training_set/images/1835_1.png', 'training_set/images/1835_2.png', 'training_set/images/1835_3.png', 'training_set/images/1835_4.png', 'training_set/images/1835_5.png', 'training_set/images/1835_6.png', 'training_set/images/1835_7.png', 'training_set/images/1835_8.png', 'training_set/images/1835_9.png', 'training_set/images/1835_10.png', 'training_set/images/1835_11.png', 'training_set/images/1835_12.png']


In [8]:
# setup ollama client: 

from ollama import Client

ollama = Client(host="http://ollama.runai-shared.svc.cluster.local")

MODEL =  "qwen2.5vl:72b" #https://ollama.com/library/qwen2.5vl
MODEL_Lang = "gpt-oss:120b"# setup ollama client: 

from ollama import Client

ollama = Client(host="http://ollama.runai-shared.svc.cluster.local")

MODEL =  "qwen2.5vl:72b" #https://ollama.com/library/qwen2.5vl
MODEL_Lang = "gpt-oss:120b"

In [ ]:
#ollama.pull(MODEL)
for progress in ollama.pull(MODEL, stream=True):
    print(progress)

In [ ]:
for progress in ollama.pull(MODEL_Lang, stream=True):
    print(progress)

In [ ]:
# list models 
for mod in ollama.list().models:
    print(mod)
    print('\n')

In [15]:
prompt_format = {
    "type": "object",
    "properties": {
        "scale": {
            "type": "string",
            "enum": ["Local", "Regional", "Global"],
            # "description": (
            #     "Local: The paper focuses on a case study from a single mine.\n"
            #     "Regional: This study focuses on a region of the world. This could be subnational (a series of case studies within an area or a region within a country), "
            #     "national (the country level impact from a single country), multinational (an area that exceeds country bounds), or an area in international waters.\n"
            #     "Global: The study is a global level analysis. "
            # ),
        },
        "country_or_continent": {
            "type": "string",
            # "description": "Which country, countries or continent is this study focused?",
        },
        "specific_location": {
            "type": "string",
            # "description": "Where specifically is the study focused? I.e., Central Africa, Clarion Clipperton Zone, Global, etc.",
        },
        "mined_commodities": {
            "type": "string",
            # "description": (
            #     "List all the commodities studied (e.g., mined) in this paper (this could be one or multiple). List any specific commodities "
            #     "that are being mined or were mined in the past. For example: If the text states that the studied mine is a coal mine, then the commodity is coal. "
            #     "Polymetallic nodules in the CCZ contain: Mn, Cu, Co, Ni."
            # ),
        },
        "mining_stage": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Mineral occurrences","Exploration/Pre-operational","Operational","Closed","Rehabilitation","NA"]
            },
            "uniqueItems": True,
            # "description": (
            #     "What is the stage of mining? Multiple options may be applicable. "
            #     "Mineral occurrences: Where a known commodity is located, this could be any stage of mining (exploration, operational, closed, etc.). "
            #     "Exploration/Pre-operational: Where a commodity is known to occur, but mining activities have not commenced. "
            #     "Operational: Mining activities are occurring. The mine is active (e.g., production reported). "
            #     "Closed: Mining activities have concluded, the site could be abandoned or rehabilitated/relinquished. "
            #     "Rehabilitation: Treatment or management of land/water disturbed by mining. Mining stage may also be operational and/or closed."                    
            # ),
            "default":[]
        },
        "type_of_mining_activity": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Open cut","Underground","Mountaintop removal","In-situ leaching","Placer mining","Dredging","Artisanal small-scale mining","Nodule mining","Sulphide mining","Crust mining","Quarrying", "NA"],
            },
            "uniqueItems": True,
            # "description": (
            #     "If this information is not explicitly stated in the study, select NA. Multiple options may be applicable."
            # ),
            "default": []
        },
        "experimental_design": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Mining area vs non-mining area","Current mining vs post-mining","Pre-mining vs current mining","Pre-mining vs post-mining","Pre-mining vs hypothetical mining", "NA"],
            },
            "uniqueItems": True,
            # "description": (
            #     "What comparison is made in mining conditions to determine biodiversity impacts? "
            #     "Mining area vs non-mining area: The impacts of mining in a mining area (any stage) are compared to an untouched area not used for mining. "
            #     "Current mining vs post-mining: The impacts of mining in an area currently being mined are compared to a different area that has undergone mining in the past. "
            #     "Pre-mining vs current mining: The impacts of current mining activities are compared to the state of biodiversity before mining commenced in the same site/area. "
            #     "Pre-mining vs post-mining: The impacts of mining on biodiversity are compared between an area that has undergone mining (after operations have ceased) and the same area before mining occurred. "
            #     "Pre-mining vs hypothetical mining: A mining area (pre-operation) is compared to a hypothetical current/post-mining scenario. "
            #     "Multiple options may be applicable."
            # ),
            "default":[]
        },
        "taxonomic_groups": {
            "type": "string",
            # "description": (
            #     "Which species group/s are investigated in this study? "
            #     "Please be specific, if possible (i.e., salamanders, skinks), otherwise use broader taxonomic group (i.e., mammals, amphibians, etc.)."
            # ),
        },
        "biomes_of_assessment": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Marine", "Freshwater", "Grassland", "Savanna", "Shrubland", "Wetlands", "Rocky areas", "Caves and subterranean", "Forest", "Desert", "Tundra", "NA"]
            },
            "uniqueItems": True,
            # "description": (
            #     "Select the biome in which the biodiversity assessment was conducted (multiple options may be applicable). If this information is not explicitly stated in the study, select NA."
            # ),
            "default": []
        },
        "method_of_assessment": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Spatial modelling","Expert opinion","Statistical analysis","In-situ sampling","Laboratory experiment","Laboratory analysis", "Interviews and questionnaires", "NA"],
            },
            "uniqueItems": True,
            # "description": (
            #     "What method is used to determine the impact or pressure on biodiversity from mining? Multiple options may be applicable. "
            #     "Spatial modelling: A spatial analysis where the mining data is intersected or overlayed with the biodiversity indicator (use of mapping software required for this selection). "
            #     "Expert opinion: The impacts to biodiversity are determined from an expert workshop or from expert judgement. "
            #     "Statistical modelling: Analysis of quantitative data to formulate results. "
            #     "In-situ sampling: Sampling is completed in two or more locations, which can be both inside and outside the impacted area, to compare biodiversity values. "
            #     "Laboratory experiment: Experiments are conducted in a laboratory setting to simulate the impacts of mining on biodiversity. "
            #     "Laboratory analysis: A lab setting is used to assess biodiversity data collected from the field and produce results. "
            #     "Interviews and questionnaires: The biodiversity impacts are determined from questionnaires or interviews with individuals from local communities, through their observations of environmental change."
            # ),
            "default":[]
        },
        "specific_methodology": {
            "type": "string",
            # "description": "Please detail the specific methodology used in the study to quantify the indirect impacts of mining on biodiversity.",
        },
        "temporal_scale": {
            "type": "string",
            "enum": ["1-5", "5-20", "20-50", "50-100", "100-1000", "1000+","NA"],
            # "description": "Over what time period (years) have the impacts to biodiversity been measured? Select 0 if measurements were taken only once. Select NA if no temporal scale is provided.",
        },
        "type_of_impact_or_pressure": {
                "type": "string",
                "enum": ["Pressure","Future impact","Observed impact"],
        },
        "impact_pathway": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Land/Seabed use","Water depletion","Ecotoxicity","Acidification","Climate change","Eutrophication","NA"],
            },
            "uniqueItems": True,
            "default": []
        },
        "description_of_impact_or_pressure": {
            "type": "string",
        },
        "spatial_scale": {
            "type": "string",
            "enum": ["0-1", "1-5", "5-10", "10-20", "20-50", "50-100", "100+","NA"],
        }
    },
    "required": [
        "scale","country_or_continent","specific_location","mined_commodities","mining_stage","type_of_mining_activity",
        "experimental_design",
        "taxonomic_groups","biomes_of_assessment","method_of_assessment","specific_methodology","temporal_scale", 
        "type_of_impact_or_pressure", "impact_pathway",
        "description_of_impact_or_pressure","spatial_scale"
    ],
    "additionalProperties": False
}


json_format_mines = {
    "type": "object",
    "properties": {
        "mine_names": {
            "type": "array", 
            "items": {"type": "string"},
            "description": (
                "List of mines that are individually assessed in the study. ",
                "To clarify: Each mine listed must be a single mine for which impacts are estimated and presented individually in the paper. ")
        }
    },
    "required": ["mine_names"],
    "additionalProperties": False
}

prompt_format_metrics = {
    "type": "array",
    "description": "List each metric used in the study to measure the impacts of mining.",
    "items": {
        "type": "object",
        "properties": {
            "metric": {
                "type": "string",
                "description": "Metric used in the study to measure the impacts of mining."
            },
            "metric_category": {
                "type": "string",
                "enum": ["Biodiversity metric","(Sub)-organismal metric","Environmental metric"],
                "description": "Metric category."
            },
            "impact_direction": {
                "type": "string",
                "enum": ["No Impact","Increased","Decreased","Changed","Variable","NA"],
                "description": (
                    "No Impact: Relative to a reference, mining has caused neither a gain nor loss to biodiversity values (For rehabilitation studies: rehabilitation has increased biodiversity values to be equivalent to reference site/s or pre-mining conditions). "
                    "Increased: Relative to a reference, mining is associated with an increase in the metric (For rehabilitation studies: rehabilitation has increased biodiversity values above reference site/s or pre-mining conditions). "
                    "Decreased: Relative to a reference, mining is associated with a decrease in the metric (For rehabilitation studies: 1) rehabilitation has increased biodiversity values but not above reference site/s or pre-mining conditions. 2) Rehab has decreased biodiversity values or is having no positive impact). "
                    "Changed: The impacts of this metric cannot be measured as increased or decreased but there has been a change when comparing between different conditions. "
                    "Variable: The impacts are reported as both increasing biodiversity values in some respects and decreasing them in others. For example, if the paper assesses different species individually and the direction is increased for one species and decreased for another."
                ),
            },
        },
        "required": [
            "metric", "metric_category", "impact_direction"
        ],
        "additionalProperties": False
    }
}


prompt_format_single_metric = {
    "type": "object",
    "properties": {
        "metric": {
            "type": "string",
            "description": "Metric used in the study to measure the impacts of mining."
        },
        "impact_direction": {
            "type": "string",
            "enum": ["No Impact","Increased","Decreased","Changed","Variable","NA"],
            "description": (
                "No Impact: Relative to a reference, mining has caused neither a gain nor loss to biodiversity values (For rehabilitation studies: rehabilitation has increased biodiversity values to be equivalent to reference site/s or pre-mining conditions). "
                "Increased: Relative to a reference, mining is associated with an increase in the metric (For rehabilitation studies: rehabilitation has increased biodiversity values above reference site/s or pre-mining conditions). "
                "Decreased: Relative to a reference, mining is associated with a decrease in the metric (For rehabilitation studies: 1) rehabilitation has increased biodiversity values but not above reference site/s or pre-mining conditions. 2) Rehab has decreased biodiversity values or is having no positive impact). "
                "Changed: The impacts of this metric cannot be measured as increased or decreased but there has been a change when comparing between different conditions. "
                "Variable: The impacts are reported as both increasing biodiversity values in some respects and decreasing them in others. For example, if the paper assesses different species individually and the direction is increased for one species and decreased for another."
            ),
        },
        "significance": {
            "type": "string",
            "enum": ["Significant","Non-significant", "NA"],
            "description": (
                "Is the change in metric stated to be significant (p value < 0.05) or non-significant? Choose NA if this is not stated in the text."
            ),
        },
        "p_value": {
            "type": "string",
            "description": (
                "What is the p-value associated with the metric change? Choose 'NA' if this is not reported in the text."
            ),
        },
        "effect_size": {
            "type": "string",
            "description": (
                "What is the effect size associated with the metric change? Choose 'NA' if this is not reported in the text."
            ),
        }
    },
    "required": [
        "metric", "impact_direction", "significance", "p_value", "effect_size"
    ],
    "additionalProperties": False
}


In [16]:
with open("system_prompt.txt", newline=None) as f:
    system_prompt = f.read()

with open("text_prompt.txt", newline=None) as f:
    text_prompt = f.read()

with open("text_prompt_mine.txt", newline=None) as f:
    text_prompt_mine = f.read()
    
with open("text_prompt_diversity_metrics.txt", newline=None) as f:
    text_prompt_diversity_metrics = f.read()

with open("text_prompt_diversity_metrics_2.txt", newline=None) as f:
    text_prompt_diversity_metrics_2 = f.read()

import importlib
import functions
importlib.reload(functions)

<module 'functions' from '/daan/Chloe-project/functions.py'>

<module 'functions' from '/daan/Chloe-project/functions.py'>

In [72]:
# df_full['Study ID'] = df_full['Study ID'].astype(str)
# df_test = df_full[df_full['Study ID'].isin(study_test)]
# df_test.to_excel("output-test.xlsx", index=False)

In [17]:
# ----------------------------------------------------------------
# Excel setup

all_cols = ["Scale", "Country/Continent", "Specific location", "Mined commodities", "Mining stage", "Type of mining activity", "Experimental design",
"Taxonomic groups", "Biomes of assessment","Method of assessment","Specific methodology", "Temporal scale", "Type of impact or pressure", 
"Impact pathway", "Description of impact or pressure", "Spatial scale", "Metric", "Metric category", "Impact direction", "Significance", 
"P value", "Effect size", "Response study", "Response metric", "Extracted tables", "Study ID"]

output_file = Path("output-rest_final.xlsx")
if output_file.exists():
    wb = load_workbook(output_file)
    ws = wb.active
else:
    wb = Workbook()
    ws = wb.active
    ws.append(list(all_cols))
    wb.save(output_file)# ----------------------------------------------------------------
# Excel setup

all_cols = ["Scale", "Country/Continent", "Specific location", "Mined commodities", "Mining stage", "Type of mining activity", "Experimental design",
"Taxonomic groups", "Biomes of assessment","Method of assessment","Specific methodology", "Temporal scale", "Type of impact or pressure", 
"Impact pathway", "Description of impact or pressure", "Spatial scale", "Metric", "Metric category", "Impact direction", "Significance", 
"P value", "Effect size", "Response study", "Response metric", "Extracted tables", "Study ID"]

output_file = Path("output-rest_final.xlsx")
if output_file.exists():
    wb = load_workbook(output_file)
    ws = wb.active
else:
    wb = Workbook()
    ws = wb.active
    ws.append(list(all_cols))
    wb.save(output_file)

In [18]:
study_train = ["2544","1825","1835","1986","2063","2067","2546","2678","2714","2758"]
study_test = ["1879", "1923", "2032", "2105", "2135", "2298", "2570", "2601", "2643", "2651", "2754", "2846", "2890", "3197", "3436"]
study_rest = ["1829", "1843", "1854", "1859", "1874", "1884", "1885", "1894", "1898", "1926", "1927", "1952", "1960", "1969", "2010", "2016", "2017", "2028", "2049", "2051", "2084", "2095", "2100", "2127", "2139", "2153", "2157", "2165", "2185", "2193", "2229", "2232", "2243", "2244", "2245", "2261", "2271", "2279", "2285", "2289", "2296", "2307", "2308", "2320", "2325", "2327", "2332", "2335", "2337", "2370", "2378", "2381", "2404", "2410", "2433", "2447", "2448", "2452", "2471", "2472", "2483", "2507", "2524", "2530", "2531", "2541", "2550", "2569", "2574", "2581", "2595", "2598", "2602", "2607", "2614", "2616", "2617", "2618", "2621", "2622", "2636", "2637", "2645", "2646", "2649", "2652", "2659", "2668", "2673", "2680", "2681", "2691", "2696", "2703", "2711", "2721", "2724", "2727", "2750", "2760", "2787", "2793", "2823", "2841", "2858", "2869", "2883", "2895", "2909", "2932", "2970", "2997", "3033", "3069", "3082", "3085", "3133", "3170", "3174", "3176", "3247", "3262", "3288"]

len(study_train), len(study_test), len(study_rest)

(10, 15, 123)

(10, 15, 123)

In [ ]:
import sys
from datetime import datetime

class Logger(object):
    def __init__(self, filename="log.txt"):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        if message.strip():
            # Define timestamp each time a message is written
            timestamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
            message = "".join([timestamp + line for line in message.splitlines(True)])
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        pass  # needed for compatibility with sys.stdout

# Redirect print() output
sys.stdout = Logger("log-rest-final.txt")

In [ ]:
import importlib
import functions
importlib.reload(functions)

# ----------------------------------------------------------------
# Loop through abstracts
start_time = time.time()

i = 0
for study_id in study_rest:  
    print(f"Iteration: {i}\t\t Study: {study_id}")

    image_paths = functions.pdf_to_images(study_id)
    
    extracted_text = ""
    with pdfplumber.open(f"extraction_papers/{study_id}.pdf") as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:  # avoid None for empty pages
                extracted_text += text + "\n\n"  # add spacing between pages

    # markdown text extracted with the GROBID model
    with open(f"extraction_papers/{study_id}.md", 'r') as f: 
       markdown_text = f.read()

    # 1. extract tables from the text, to ensure correct interpretation of results 
    print("Extracting tables from the text...")
    text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 
    
    # 2. get names of individual mines, if assessed individually in the study: 
    print("Determining whether multiple mines are assessed individually in the study...") 
    mines_prompt = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text" + extracted_text + "\n\nTables from the text: " + text_tables + """\n\n
    If this mining-impact study individually assessed the impacts of mining of one or more single mines, list each individual mine. 
    Only list single, individual mines, don't list different study locations that assessed the same mine. To clarify: Each listed mine must be one single mine for which 
    impacts are estimated and presented individually in the paper.
    """

    mines, _ = functions.extract_information_formatted(ollama, MODEL, SEED, system_prompt, mines_prompt, json_format_mines, image_paths) 
    #mines_response, _ = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = mines_prompt)
    #mines = functions.get_json_table(ollama, MODEL_Lang, SEED, mines_response, json_format_mines)
    
    if (mines is None) or (mines=={}) or (mines.get("mine_names", [])==[]) or (len(mines.get("mine_names", []))==1): 
        print("No (multiple) individual mines found, starting extraction for complete study...")
        
        df = functions.get_full_response_json(ollama, 
                                              MODEL, 
                                              MODEL_Lang, 
                                              SEED, 
                                              study_id, 
                                              system_prompt, 
                                              text_prompt, 
                                              text_prompt_diversity_metrics, 
                                              text_prompt_diversity_metrics_2, 
                                              text_tables, 
                                              prompt_format, 
                                              prompt_format_metrics, 
                                              prompt_format_single_metric, 
                                              image_paths, 
                                              extracted_text,
                                              markdown_text
                                       )

        df = df[all_cols] # reorder columns 

        df["P value"] = (
            df["P value"]
            .astype(str)
            .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
        )
    
        # append to excel workbook 
        for j, row in enumerate(df.itertuples(index=False), start=1):
            ws.append(row)
            
        print("Done.")
        
    else: 
        print(f"Multiple mines found: {', '.join(mines.get("mine_names"))}")

        for mine in mines.get("mine_names"):
            print(f"Starting extraction for mine: {mine}...")
            
            df = functions.get_full_response_json(ollama, 
                                                  MODEL, 
                                                  MODEL_Lang, 
                                                  SEED, 
                                                  study_id, 
                                                  system_prompt, 
                                                  text_prompt_mine.format(mine=mine), 
                                                  text_prompt_diversity_metrics,
                                                  text_prompt_diversity_metrics_2, 
                                                  text_tables, 
                                                  prompt_format, 
                                                  prompt_format_metrics, 
                                                  prompt_format_single_metric, 
                                                  image_paths, 
                                                  extracted_text,
                                                  markdown_text,
                                                  prompt_prefix = f"For the mine named '{mine}' specifically, do the following:"
                                                 )

            df['Scale'] = "Local" # per definition 
            df = df[all_cols] # reorder columns 

            df["P value"] = (
                df["P value"]
                .astype(str)
                .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
            )
    
            # append to excel workbook 
            for j, row in enumerate(df.itertuples(index=False), start=1):
                ws.append(row)
                
            print("Done.")
            
    # make sure all elements are strings 
    #json_table = json_table.map(lambda x: ", ".join(x) if isinstance(x, list) else x)

    wb.save(output_file)
    i+=1

elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")import importlib
import functions
importlib.reload(functions)

# ----------------------------------------------------------------
# Loop through abstracts
start_time = time.time()

i = 0
for study_id in study_rest:  
    print(f"Iteration: {i}\t\t Study: {study_id}")

    image_paths = functions.pdf_to_images(study_id)
    
    extracted_text = ""
    with pdfplumber.open(f"extraction_papers/{study_id}.pdf") as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:  # avoid None for empty pages
                extracted_text += text + "\n\n"  # add spacing between pages

    # markdown text extracted with the GROBID model
    with open(f"extraction_papers/{study_id}.md", 'r') as f: 
       markdown_text = f.read()

    # 1. extract tables from the text, to ensure correct interpretation of results 
    print("Extracting tables from the text...")
    text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 
    
    # 2. get names of individual mines, if assessed individually in the study: 
    print("Determining whether multiple mines are assessed individually in the study...") 
    mines_prompt = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text" + extracted_text + "\n\nTables from the text: " + text_tables + """\n\n
    If this mining-impact study individually assessed the impacts of mining of one or more single mines, list each individual mine. 
    Only list single, individual mines, don't list different study locations that assessed the same mine. To clarify: Each listed mine must be one single mine for which 
    impacts are estimated and presented individually in the paper.
    """

    mines, _ = functions.extract_information_formatted(ollama, MODEL, SEED, system_prompt, mines_prompt, json_format_mines, image_paths) 
    #mines_response, _ = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = mines_prompt)
    #mines = functions.get_json_table(ollama, MODEL_Lang, SEED, mines_response, json_format_mines)
    
    if (mines is None) or (mines=={}) or (mines.get("mine_names", [])==[]) or (len(mines.get("mine_names", []))==1): 
        print("No (multiple) individual mines found, starting extraction for complete study...")
        
        df = functions.get_full_response_json(ollama, 
                                              MODEL, 
                                              MODEL_Lang, 
                                              SEED, 
                                              study_id, 
                                              system_prompt, 
                                              text_prompt, 
                                              text_prompt_diversity_metrics, 
                                              text_prompt_diversity_metrics_2, 
                                              text_tables, 
                                              prompt_format, 
                                              prompt_format_metrics, 
                                              prompt_format_single_metric, 
                                              image_paths, 
                                              extracted_text,
                                              markdown_text
                                       )

        df = df[all_cols] # reorder columns 

        df["P value"] = (
            df["P value"]
            .astype(str)
            .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
        )
    
        # append to excel workbook 
        for j, row in enumerate(df.itertuples(index=False), start=1):
            ws.append(row)
            
        print("Done.")
        
    else: 
        print(f"Multiple mines found: {', '.join(mines.get("mine_names"))}")

        for mine in mines.get("mine_names"):
            print(f"Starting extraction for mine: {mine}...")
            
            df = functions.get_full_response_json(ollama, 
                                                  MODEL, 
                                                  MODEL_Lang, 
                                                  SEED, 
                                                  study_id, 
                                                  system_prompt, 
                                                  text_prompt_mine.format(mine=mine), 
                                                  text_prompt_diversity_metrics,
                                                  text_prompt_diversity_metrics_2, 
                                                  text_tables, 
                                                  prompt_format, 
                                                  prompt_format_metrics, 
                                                  prompt_format_single_metric, 
                                                  image_paths, 
                                                  extracted_text,
                                                  markdown_text,
                                                  prompt_prefix = f"For the mine named '{mine}' specifically, do the following:"
                                                 )

            df['Scale'] = "Local" # per definition 
            df = df[all_cols] # reorder columns 

            df["P value"] = (
                df["P value"]
                .astype(str)
                .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
            )
    
            # append to excel workbook 
            for j, row in enumerate(df.itertuples(index=False), start=1):
                ws.append(row)
                
            print("Done.")
            
    # make sure all elements are strings 
    #json_table = json_table.map(lambda x: ", ".join(x) if isinstance(x, list) else x)

    wb.save(output_file)
    i+=1

elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")

In [ ]:
# Restore normal printing
sys.stdout.log.close()
sys.stdout = sys.stdout.terminal

# Restore the original stdout so print() shows in the notebook again
if hasattr(sys.stdout, "terminal"):
    sys.stdout = sys.stdout.terminal

In [ ]:
# Finished in 106.99 minutes

In [ ]:
# ----------------------------------------------------------------
# Excel setup

all_cols = ["Scale", "Country/Continent", "Specific location", "Mined commodities", "Mining stage", "Type of mining activity", "Experimental design",
"Taxonomic groups", "Biomes of assessment","Method of assessment","Specific methodology", "Temporal scale", "Type of impact or pressure", 
"Impact pathway", "Description of impact or pressure", "Spatial scale", "Metric", "Metric category", "Impact direction", "Significance", 
"P value", "Effect size", "Response study", "Response metric", "Extracted tables", "Study ID"]

output_file = Path("output-test_final.xlsx")
if output_file.exists():
    wb = load_workbook(output_file)
    ws = wb.active
else:
    wb = Workbook()
    ws = wb.active
    ws.append(list(all_cols))
    wb.save(output_file)

# Redirect print() output
sys.stdout = Logger("log-test-final.txt")

import importlib
import functions
importlib.reload(functions)

# ----------------------------------------------------------------
# Loop through abstracts
start_time = time.time()

i = 0
for study_id in study_test:  
    print(f"Iteration: {i}\t\t Study: {study_id}")

    image_paths = functions.pdf_to_images(study_id)
    
    extracted_text = ""
    with pdfplumber.open(f"extraction_papers/{study_id}.pdf") as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:  # avoid None for empty pages
                extracted_text += text + "\n\n"  # add spacing between pages

    # markdown text extracted with the GROBID model
    with open(f"extraction_papers/{study_id}.md", 'r') as f: 
       markdown_text = f.read()

    # 1. extract tables from the text, to ensure correct interpretation of results 
    print("Extracting tables from the text...")
    text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 
    
    # 2. get names of individual mines, if assessed individually in the study: 
    print("Determining whether multiple mines are assessed individually in the study...") 
    mines_prompt = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text" + extracted_text + "\n\nTables from the text: " + text_tables + """\n\n
    If this mining-impact study individually assessed the impacts of mining of one or more single mines, list each individual mine. 
    Only list single, individual mines, don't list different study locations that assessed the same mine. To clarify: Each listed mine must be one single mine for which 
    impacts are estimated and presented individually in the paper.
    """

    mines, _ = functions.extract_information_formatted(ollama, MODEL, SEED, system_prompt, mines_prompt, json_format_mines, image_paths) 
    #mines_response, _ = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = mines_prompt)
    #mines = functions.get_json_table(ollama, MODEL_Lang, SEED, mines_response, json_format_mines)
    
    if (mines is None) or (mines=={}) or (mines.get("mine_names", [])==[]) or (len(mines.get("mine_names", []))==1): 
        print("No (multiple) individual mines found, starting extraction for complete study...")
        
        df = functions.get_full_response_json(ollama, 
                                              MODEL, 
                                              MODEL_Lang, 
                                              SEED, 
                                              study_id, 
                                              system_prompt, 
                                              text_prompt, 
                                              text_prompt_diversity_metrics, 
                                              text_prompt_diversity_metrics_2, 
                                              text_tables, 
                                              prompt_format, 
                                              prompt_format_metrics, 
                                              prompt_format_single_metric, 
                                              image_paths, 
                                              extracted_text,
                                              markdown_text
                                       )

        df = df[all_cols] # reorder columns 

        df["P value"] = (
            df["P value"]
            .astype(str)
            .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
        )
    
        # append to excel workbook 
        for j, row in enumerate(df.itertuples(index=False), start=1):
            ws.append(row)
            
        print("Done.")
        
    else: 
        print(f"Multiple mines found: {', '.join(mines.get("mine_names"))}")

        for mine in mines.get("mine_names"):
            print(f"Starting extraction for mine: {mine}...")
            
            df = functions.get_full_response_json(ollama, 
                                                  MODEL, 
                                                  MODEL_Lang, 
                                                  SEED, 
                                                  study_id, 
                                                  system_prompt, 
                                                  text_prompt_mine.format(mine=mine), 
                                                  text_prompt_diversity_metrics,
                                                  text_prompt_diversity_metrics_2, 
                                                  text_tables, 
                                                  prompt_format, 
                                                  prompt_format_metrics, 
                                                  prompt_format_single_metric, 
                                                  image_paths, 
                                                  extracted_text,
                                                  markdown_text,
                                                  prompt_prefix = f"For the mine named '{mine}' specifically, do the following:"
                                                 )

            df['Scale'] = "Local" # per definition 
            df = df[all_cols] # reorder columns 

            df["P value"] = (
                df["P value"]
                .astype(str)
                .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
            )
    
            # append to excel workbook 
            for j, row in enumerate(df.itertuples(index=False), start=1):
                ws.append(row)
                
            print("Done.")
            
    # make sure all elements are strings 
    #json_table = json_table.map(lambda x: ", ".join(x) if isinstance(x, list) else x)

    wb.save(output_file)
    i+=1

elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")

# Restore normal printing
sys.stdout.log.close()
sys.stdout = sys.stdout.terminal

# Restore the original stdout so print() shows in the notebook again
if hasattr(sys.stdout, "terminal"):
    sys.stdout = sys.stdout.terminal

[2025-12-10 20:11:23] Iteration: 0		 Study: 1879
[2025-12-10 20:11:57] Extracting tables from the text...
[2025-12-10 20:31:49] Done.
[2025-12-10 20:31:49] Iteration: 1		 Study: 1923
[2025-12-10 20:32:14] Extracting tables from the text...
[2025-12-10 20:34:36] Determining whether multiple mines are assessed individually in the study...
[2025-12-10 20:36:14] No (multiple) individual mines found, starting extraction for complete study...
[2025-12-10 20:39:00] Extracting information for metric: Total mercury concentration (THg) in pool water + sedimentary organic matter (ppm)
[2025-12-10 20:39:30] Extracting information for metric: Proportion of pools with Hg > Severe Effect Level (SEL = 2 ppm)
[2025-12-10 20:39:56] Extracting information for metric: Distance from each study site/pool to the nearest known ASGM site (km)
[2025-12-10 20:40:28] Extracting information for metric: Presence/absence of Dendrobates tinctorius tadpoles in a pool (binary)
[2025-12-10 20:40:55] Extracting informati

In [432]:
#### Example: Study 1835 #### 

import importlib
import functions
importlib.reload(functions)

study_id = "1835"

print(f"Iteration: {i}\t\t Study: {study_id}")

image_paths = functions.pdf_to_images(study_id)

extracted_text = ""
with pdfplumber.open(f"extraction_papers/{study_id}.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:  # avoid None for empty pages
            extracted_text += text + "\n\n"  # add spacing between pages

# markdown text extracted with the GROBID model
with open(f"extraction_papers/{study_id}.md", 'r') as f: 
    markdown_text = f.read()

# 1. extract tables from the text, to ensure correct interpretation of results 
print("Extracting tables from the text...")
text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 


Iteration: 0		 Study: 1835
Extracting tables from the text...


In [435]:
with open("text_prompt_diversity_metrics.txt", newline=None) as f:
    text_prompt_diversity_metrics = f.read()

with open("text_prompt_diversity_metrics_2.txt", newline=None) as f:
    text_prompt_diversity_metrics_2 = f.read()

import importlib
import functions
importlib.reload(functions)

# 2. get text response for the metrics (supply images, text and extracted tables): 
user_prompt_2 = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text: " + extracted_text + "\n\nTables from the text: " + text_tables  + "\n\n" + text_prompt_diversity_metrics

response_metrics, json_metrics = functions.extract_and_return_as_json(ollama, MODEL_Lang, SEED, system_prompt, user_prompt_2, prompt_format_metrics, images=False, image_paths=image_paths)

print(response_metrics)
json_metrics


<module 'functions' from '/daan/Chloe-project/functions.py'>

In [436]:
cols = ["Metric", "Metric category", "Impact direction", "Significance", "P value", "Effect size", "Response metric"]
df = pd.DataFrame([[""] * len(cols)],columns=cols) # empty dataframe
for metric in json_metrics: 
    df_row = pd.DataFrame([[""] * len(cols)],columns=cols) # empty dataframe
    df_row['Metric'] = metric.get('metric', "")
    df_row['Metric category'] = metric.get('metric_category', "")
    df = pd.concat([df, df_row], ignore_index=True)
df = df[1:] # first row is empty 

for m in df['Metric']: 
    print(f"Extracting information for metric: {m}")
    
    # setup prompt for metric
    user_prompt_3 = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text: " + extracted_text + "\n\nTables from the text: " + text_tables + "\n\n" + f"For the metric {m} specifically, determine:\n" + text_prompt_diversity_metrics_2
    
    # extract json 
    response_metric, json_metric = functions.extract_and_return_as_json(ollama, MODEL_Lang, SEED, system_prompt, user_prompt_3, prompt_format_single_metric, images=False, image_paths=image_paths)
    
    # add to df 
    df.loc[df['Metric'] == m, 'Response metric']  = response_metric # save text response
    if json_metric is not None: 
        if isinstance(json_metric, list): # the metric was split into a list of individual metrics  
            for metric_i in json_metric:
                df_i = pd.DataFrame([[""] * len(cols)],columns=cols)
                df_i['Metric']           = metric_i.get("metric", "")
                df_i['Metric category']  = df.loc[df['Metric'] == m, 'Metric category'].iloc[0]
                df_i['Impact direction'] = metric_i.get("impact_direction", "")
                df_i['Significance']     = metric_i.get("significance", "")
                df_i['P value']          = metric_i.get("p_value", "")
                df_i['Effect size']      = metric_i.get("effect_size", "")
                df_i['Response metric']  = response_metric # save text response
                df = pd.concat([df, df_i], ignore_index=True)
            df = df[df["Metric"] != m] # remove the earlier metric (which has been replaced) 
        else: 
            df.loc[df['Metric'] == m, 'Impact direction'] = json_metric.get("impact_direction", "")
            df.loc[df['Metric'] == m, 'Significance']     = json_metric.get("significance", "")
            df.loc[df['Metric'] == m, 'P value']          = json_metric.get("p_value", "")
            df.loc[df['Metric'] == m, 'Effect size']      = json_metric.get("effect_size", "")

df
            

Extracting information for metric: Species‑richness (total number of fish species)
Extracting information for metric: Fish‑capture abundance (average daily catch per fisher, kg)
Extracting information for metric: Fish‑size structure (proportion of large, medium and small fish)
Extracting information for metric: Observation of high fish mortality (percentage of fishers reporting mass mortality)
Extracting information for metric: Water‑temperature (°C)
Extracting information for metric: pH
Extracting information for metric: Electrical Conductivity (EC, μS cm⁻¹)
Extracting information for metric: Total Dissolved Solids (TDS, mg L⁻¹)
Extracting information for metric: Chemical Oxygen Demand (COD, mg L⁻¹)
Extracting information for metric: Magnesium concentration (mg L⁻¹)
Extracting information for metric: Sediment presence in water (qualitative “yes/no”)
Extracting information for metric: Water‑colour change (perceived effect on water quality)
Extracting information for metric: Turbidity (

,Metric,Metric category,Impact direction,Significance,P value,Effect size,Response metric
1,Species‑richness (total number of fish species),Biodiversity metric,Decreased,NA,NA,NA,"**Metric:** Species‑richness (total number of fish species) \n\n| Impact direction | Significance | P‑value | Effect size |\n|-----------------|--------------|---------|-------------|\n| **Decreased** | **NA** (no statistical test reported) | **NA** | **NA** |\n\n**Reasoning**\n\n1. **Impact direction** – The paper compares the *current* fish community (post‑mining) with the community that existed *15 years ago* (pre‑mining or pre‑intensification of mining). \n - Table 6 shows **7** species currently present versus **13** species reported 15 years ago. \n - The authors state that “only seven of these species remain, indicating partial extinction of six species” and that “fishers perceive a decline in fish yield and diversity due to mining activities.” \n - Therefore, relative to the pre‑mining reference, mining is associated with a **decrease** in species‑richness.\n\n2. **Significance** – The manuscript does not present any statistical test (e.g., t‑test, ANOVA, chi‑square) that evaluates whether the reduction from 13 to 7 species is statistically significant. Consequently, the significance level cannot be determined → **NA**.\n\n3. **P‑value** – No p‑value is reported for the change in total species number. → **NA**.\n\n4. **Effect size** – The authors give only the raw counts (13 → 7) and a qualitative description of “partial extinction”; no quantitative effect‑size metric (e.g., Cohen’s d, percent change) is provided. → **NA**."
2,"Fish‑capture abundance (average daily catch per fisher, kg)",Biodiversity metric,Decreased,NA,NA,NA,"**Metric examined:** Average daily fish‑capture abundance per fisher (kg).\n\n### 1. Impact direction \nThe paper reports the average daily catch **15 years ago (pre‑mining/intense mining period)** as **9.00 ± 4.20 kg** and the **current value (post‑mining)** as **3.62 ± 3.39 kg** (Table 2, “Quantity (in Kg 15 years ago)” vs. “Current quantity (Kg)”). \nBecause the current value is lower than the pre‑mining value, the metric is **decreased** relative to the non‑mining/pre‑mining reference.\n\n### 2. Significance of the change \nThe manuscript does not present a statistical test that directly compares the 15‑year‑ago catch with the current catch (no t‑test, ANOVA, or similar is reported for this temporal contrast). Therefore the significance of the observed decrease is **not reported** → **NA**.\n\n### 3. P‑value \nSince no test of the temporal change is given, **no p‑value** is provided for this metric → **NA**.\n\n### 4. Effect size \nThe article does not give an effect‑size measure (e.g., Cohen’s d, percent change, confidence interval) for the decline in daily catch → **NA**.\n\n---\n\n**Summary**\n\n| Impact direction | Significance | P‑value | Effect size |\n|------------------|--------------|---------|-------------|\n| Decreased | NA | NA | NA |\n\nThe decrease is described only descriptively (average 9 kg → 3.6 kg), with no accompanying statistical test or magnitude estimate."
3,"Fish‑size structure (proportion of large, medium and small fish)",Biodiversity metric,Decreased,NA,NA,NA,"**1. Impact direction** \nThe paper reports the *current* fish‑size structure (large = 16.9 %, medium = 33.7 %, small = 49.4 %) and compares it with the *status 15 years ago* (large = 49.4 %, medium = 42.2 %, small = 8.4 %). \nBecause mining has been occurring for the past 15 years, this comparison is effectively **mining vs pre‑mining**. The proportion of large fish has fallen dramatically (≈‑33 percentage points) while the proportion of small fish has risen (≈ + 41 pp). Hence the metric “fish‑size structure” shows a shift toward smaller individuals, i.e. a **decrease** in the size of fish captured relative to the pre‑mining reference. \n\n**Impact direction:** *Decreased* \n\n**2. Significance** \nThe manuscript does not provide a s